# Titanic
## Score: .76794

In [13]:
from __future__ import annotations

import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

ROOT = Path.cwd()
DATA = ROOT / "titanic"
assert (DATA / "train.csv").exists(), "Set cwd to project root (folder containing titanic/train.csv)"

CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [14]:
def extract_title(name: str) -> str:
    m = re.search(r",\s*([^\.]+)\.", name)
    return m.group(1).strip() if m else "Unknown"


def cabin_deck(cabin) -> str:
    if pd.isna(cabin) or not str(cabin).strip():
        return "U"
    c = str(cabin).strip()[0]
    return c if c.isalpha() else "U"


def add_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["Title"] = out["Name"].map(extract_title)
    rare = {
        "Lady",
        "Countess",
        "Capt",
        "Col",
        "Don",
        "Dr",
        "Major",
        "Rev",
        "Sir",
        "Jonkheer",
        "Dona",
    }
    out["Title"] = out["Title"].replace(
        {
            "Mlle": "Miss",
            "Ms": "Miss",
            "Mme": "Mrs",
        }
    )
    out.loc[out["Title"].isin(rare), "Title"] = "Rare"
    out["FamilySize"] = out["SibSp"] + out["Parch"] + 1
    out["IsAlone"] = (out["FamilySize"] == 1).astype(int)
    out["HasCabin"] = out["Cabin"].notna().astype(int)
    out["Deck"] = out["Cabin"].map(cabin_deck)
    out["FarePerPerson"] = out["Fare"] / out["FamilySize"].clip(lower=1)
    out["LogFare"] = np.log1p(out["Fare"])
    out["IsChild"] = np.where(out["Age"].notna(), (out["Age"] < 16).astype(int), 0)
    return out

In [15]:
def build_pipeline() -> Pipeline:
    numeric = [
        "Pclass",
        "Age",
        "SibSp",
        "Parch",
        "Fare",
        "FamilySize",
        "IsAlone",
        "HasCabin",
        "FarePerPerson",
        "LogFare",
        "IsChild",
    ]
    categorical = ["Sex", "Embarked", "Title", "Deck"]

    pre = ColumnTransformer(
        [
            ("num", SimpleImputer(strategy="median"), numeric),
            (
                "cat",
                Pipeline(
                    [
                        ("impute", SimpleImputer(strategy="most_frequent")),
                        ("oh", OneHotEncoder(handle_unknown="ignore")),
                    ]
                ),
                categorical,
            ),
        ]
    )

    clf = RandomForestClassifier(
        n_estimators=500,
        max_depth=10,
        min_samples_leaf=2,
        min_samples_split=4,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1,
    )
    return Pipeline([("prep", pre), ("model", clf)])

In [16]:
train = pd.read_csv(DATA / "train.csv")
test = pd.read_csv(DATA / "test.csv")

train_x = add_features(train.drop(columns=["Survived"]))
train_y = train["Survived"]
test_x = add_features(test)

pipe = build_pipeline()
cv_scores = cross_val_score(pipe, train_x, train_y, cv=CV, scoring="accuracy", n_jobs=-1)
print(f"RF CV accuracy: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}")

pipe.fit(train_x, train_y)
pred = pipe.predict(test_x)
sub = pd.DataFrame({"PassengerId": test["PassengerId"], "Survived": pred.astype(int)})
out_path = ROOT / "submission.csv"
sub.to_csv(out_path, index=False)
print(f"Wrote {out_path} ({len(sub)} rows)")
sub.head(10)

RF CV accuracy: 0.8395 +/- 0.0106
Wrote c:\Users\ol1v3_7dwns5u\OneDrive\Desktop\ACTIVE\Titanic\submission.csv (418 rows)


,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1
5,897,0
6,898,1
7,899,0
8,900,1
9,901,0


In [17]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import cross_val_predict

# OOF predictions: each row predicted when it was in the holdout fold (same CV as above).
oof_pred = cross_val_predict(pipe, train_x, train_y, cv=CV, n_jobs=-1)
oof_proba = cross_val_predict(pipe, train_x, train_y, cv=CV, method="predict_proba", n_jobs=-1)[:, 1]

print("Out-of-fold on training set (same StratifiedKFold as CV accuracy)")
print(classification_report(train_y, oof_pred, digits=4))
print("Confusion matrix [ [TN FP], [FN TP] ]:")
print(confusion_matrix(train_y, oof_pred))
print(f"ROC-AUC (OOF): {roc_auc_score(train_y, oof_proba):.4f}")

prep = pipe.named_steps["prep"]
feat_names = prep.get_feature_names_out()
imp = pd.Series(pipe.named_steps["model"].feature_importances_, index=feat_names).sort_values(ascending=False)
print("\nTop feature importances (model refit on full train):")
print(imp.head(15).to_string())

Out-of-fold on training set (same StratifiedKFold as CV accuracy)
              precision    recall  f1-score   support

           0     0.8464    0.9035    0.8740       549
           1     0.8262    0.7368    0.7790       342

    accuracy                         0.8395       891
   macro avg     0.8363    0.8202    0.8265       891
weighted avg     0.8387    0.8395    0.8375       891

Confusion matrix [ [TN FP], [FN TP] ]:
[[496  53]
 [ 90 252]]
ROC-AUC (OOF): 0.8799

Top feature importances (model refit on full train):
cat__Sex_female       0.120634
cat__Sex_male         0.115957
cat__Title_Mr         0.114749
num__FarePerPerson    0.096377
num__Fare             0.078349
num__LogFare          0.076431
num__Age              0.074482
num__Pclass           0.055537
num__FamilySize       0.037617
cat__Title_Miss       0.036770
cat__Title_Mrs        0.031410
num__SibSp            0.025127
cat__Deck_U           0.022482
num__HasCabin         0.018594
num__Parch            0.013080
